In [1]:
import fitz  # PyMuPDF
import re
import pandas as pd

In [11]:
pdf_path = "MICO.pdf"

# --- Expresión regular mejorada ---
pattern = re.compile(
    r"([A-Z0-9-]+)\s+"              # Doc/Factura
    r"(\d+)\s+"                     # No.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Contabiliz.
    r"([\d.,]+-?)\s+"               # Importe (puede terminar en '-')
    r"([A-Z]{3})"                   # Moneda
)

rows = []

with fitz.open(pdf_path) as doc:
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        matches = pattern.findall(text)
        print(f"📄 Página {page_num} — {len(matches)} filas detectadas")
        for m in matches:
            rows.append(m)

# --- Crear DataFrame ---
df = pd.DataFrame(
    rows,
    columns=[
        "Doc/Factura",
        "No.Documento",
        "Fc.Documento",
        "Fc.Contabiliz.",
        "Importe",
        "Moneda",
    ],
)

# --- Limpieza de importes mejorada ---
def parse_importe(val):
    """
    Convierte valores tipo '27,024.63-' o '895.76-' a floats con signo correcto.
    """
    if not isinstance(val, str):
        return None
    val = val.strip()
    negativo = val.endswith("-")
    val = val.replace("-", "")
    
    # Si el número tiene ambos ',' y '.', asumimos que ',' son miles y '.' son decimales.
    if "," in val and "." in val:
        val = val.replace(",", "")
    # Si tiene solo ',', asumimos que ',' son decimales (caso latinoamericano)
    elif "," in val and "." not in val:
        val = val.replace(",", ".")
        
    try:
        num = float(val)
        return -num if negativo else num
    except ValueError:
        return None

df["Importe"] = df["Importe"].apply(parse_importe)

print(f"\n✅ Total filas encontradas: {len(df)}")
display(df.head(15))


📄 Página 1 — 34 filas detectadas
📄 Página 2 — 28 filas detectadas

✅ Total filas encontradas: 62


,Doc/Factura,No.Documento,Fc.Documento,Fc.Contabiliz.,Importe,Moneda
0,013002-000333758,2101018298,08.07.2025,09.07.2025,-27024.63,USD
1,013002-000334721,2101103158,23.07.2025,24.07.2025,-895.76,USD
2,013002-000335393,2101143618,30.07.2025,31.07.2025,-15874.06,USD
3,013002-000335645,2101168504,04.08.2025,05.08.2025,-41992.26,USD
4,013002-000335767,2101182754,06.08.2025,07.08.2025,-32212.92,USD
5,013002-000335763,2101182761,06.08.2025,07.08.2025,-8195.59,USD
6,013002-000335762,2101182762,06.08.2025,07.08.2025,-16499.86,USD
7,013002-000335766,2101182787,06.08.2025,07.08.2025,-41162.44,USD
8,013002-000335764,2101182788,06.08.2025,07.08.2025,-27950.46,USD
9,013002-000335765,2101182789,06.08.2025,07.08.2025,-17952.78,USD


In [14]:
import fitz  # PyMuPDF
import re
import pandas as pd

pdf_path = "MICO.pdf"

# --- Expresión regular robusta ---
pattern = re.compile(
    r"([A-Z0-9\-]+)\s+"             # Doc/Factura
    r"(\d+)\s+"                     # No.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Documento
    r"(\d{2}\.\d{2}\.\d{4})\s+"     # Fc.Contabiliz.
    r"([\d.,]+-?)\s+"               # Importe (puede terminar en '-')
    r"([A-Z]{3})"                   # Moneda
)

rows = []

with fitz.open(pdf_path) as doc:
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        matches = pattern.findall(text)
        print(f"📄 Página {page_num} — {len(matches)} filas detectadas")
        for m in matches:
            rows.append(m)

# --- Crear DataFrame ---
df = pd.DataFrame(
    rows,
    columns=[
        "Doc/Factura",
        "No.Documento",
        "Fc.Documento",
        "Fc.Contabiliz.",
        "Importe",
        "Moneda",
    ],
)

# --- Limpieza de importes (siempre positivos) ---
def parse_importe(val):
    """
    Convierte valores tipo '27,024.63-' o '895.76-' en floats POSITIVOS (sin signo negativo).
    """
    if not isinstance(val, str):
        return None
    val = val.strip().replace("-", "")  # Quitamos el signo

    # Si hay ',' y '.', asumimos formato tipo '27,024.63' → quitamos ',' (miles)
    if "," in val and "." in val:
        val = val.replace(",", "")
    # Si hay solo ',', asumimos que ',' son decimales (formato europeo)
    elif "," in val and "." not in val:
        val = val.replace(",", ".")

    try:
        return float(val)
    except ValueError:
        return None

df["Importe"] = df["Importe"].apply(parse_importe)

print(f"\n✅ Total filas encontradas: {len(df)}")
display(df.head(15))


📄 Página 1 — 34 filas detectadas
📄 Página 2 — 28 filas detectadas

✅ Total filas encontradas: 62


,Doc/Factura,No.Documento,Fc.Documento,Fc.Contabiliz.,Importe,Moneda
0,013002-000333758,2101018298,08.07.2025,09.07.2025,27024.63,USD
1,013002-000334721,2101103158,23.07.2025,24.07.2025,895.76,USD
2,013002-000335393,2101143618,30.07.2025,31.07.2025,15874.06,USD
3,013002-000335645,2101168504,04.08.2025,05.08.2025,41992.26,USD
4,013002-000335767,2101182754,06.08.2025,07.08.2025,32212.92,USD
5,013002-000335763,2101182761,06.08.2025,07.08.2025,8195.59,USD
6,013002-000335762,2101182762,06.08.2025,07.08.2025,16499.86,USD
7,013002-000335766,2101182787,06.08.2025,07.08.2025,41162.44,USD
8,013002-000335764,2101182788,06.08.2025,07.08.2025,27950.46,USD
9,013002-000335765,2101182789,06.08.2025,07.08.2025,17952.78,USD
